# Demo Pemrosesan CV dengan LLM
## Mini Assignment Sekolah Karir

Notebook ini mendemonstrasikan ekstraksi informasi CV menjadi profil terstruktur, pengecekan bukti pada teks sumber, ringkasan kandidat, pencocokan kebutuhan pekerjaan, dan rekomendasi untuk ditinjau manusia.

**Data demo:** lima CV fiktif di `data/cv/` (lihat `data/TEST_MANIFEST.json`) dan kebutuhan posisi **Junior AI Engineer** di `data/jobs/ai_engineer.json`.

> Salinan ini menambahkan dokumentasi Markdown saja. Seluruh cell code, urutannya, metadata cell, execution count, dan output tersimpan dipertahankan dari `demo.ipynb`. Output tersimpan merupakan hasil eksekusi sebelumnya; dokumen ini tidak mengeksekusi ulang notebook.

### Persiapan dan cara menjalankan

1. Gunakan Python **3.12 atau lebih baru** dan environment proyek dengan dependensi dari `pyproject.toml`, termasuk `ipykernel` untuk notebook. Jika menggunakan uv, jalankan `uv sync --group dev` dari root repository.
2. Pilih kernel environment proyek dan pastikan working directory berada di root repository agar path `data/...` dapat ditemukan.
3. Siapkan `OPENAI_API_KEY` melalui environment atau `.env`. Modul ekstraksi memuat `.env`; `OPENAI_MODEL` dapat dipakai untuk memilih model, dengan default di kode `gpt-4o-mini`.
4. Jalankan cell dari atas ke bawah. Cell berikutnya menggunakan variabel dari cell sebelumnya. Ekstraksi, overview, dan rekomendasi memanggil API OpenAI sehingga memerlukan koneksi dan dapat menimbulkan biaya; evaluasi batch mengulang pemanggilan untuk setiap CV.

### Peta notebook

| Bagian | Isi |
| --- | --- |
| 1. Persiapan data | Membaca PDF dan menyamarkan informasi pribadi |
| 2. Ekstraksi dan bukti | Profil terstruktur, evidence coverage, dan percobaan skill tambahan |
| 3. Ringkasan dan pencocokan | Overview, kebutuhan pekerjaan, rekomendasi, dan assertion |
| 4. Evaluasi batch | Pemrosesan lima CV, tabel hasil, dan metrik ekstraksi skill |
| 5. Pemeriksaan tambahan | Masking, aturan rekomendasi, dan konfirmasi kebutuhan pekerjaan |

**Alur data:** PDF → teks → masking → profil kandidat → validasi bukti → overview dan matching → rekomendasi dengan human review.

## 1. Persiapan data
### 1.1 Membaca teks dari PDF
Memilih CV Nadia sebagai contoh awal, lalu menyimpan teksnya dalam `cv_text` melalui modul `parser.py`. Parser menggunakan ekstraksi teks PDF, bukan OCR, sehingga PDF hasil scan tanpa lapisan teks belum tercakup oleh langkah ini.

Baris untuk menampilkan cuplikan teks masih berupa komentar, sesuai notebook asli.

In [1]:
from pathlib import Path
from mini_assignment_sekolahkarir.parser import extract_text_from_pdf

pdf_path = Path("data/cv/CV_01_Nadia_Putri_AI_Engineer.pdf")

cv_text = extract_text_from_pdf(pdf_path)

# print(cv_text[:2000])

c:\Users\fazam\OneDrive\Documents\Developer Workspace\01-personal-projects\active\mini-assignment-sekolahkarir\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


### 1.2 Menyamarkan informasi pribadi
`mask_personal_information` mengganti pola informasi pribadi yang dikenali menjadi placeholder. Hasilnya disimpan sebagai `masked_cv`; output menampilkan maksimal 3.000 karakter pertama.

Masking berbasis aturan di `privacy.py` mencakup email, telepon, NIK, serta pola atau label alamat, tanggal lahir, dan kontak darurat. Ini bukan jaminan anonimisasi lengkap; nama dan informasi profesional dapat tetap muncul.

In [2]:
from mini_assignment_sekolahkarir.privacy import (
    mask_personal_information
)

masked_cv = mask_personal_information(cv_text)

print(masked_cv[:3000])

NADIA PUTRI MAHARANI
 [PHONE_REDACTED] | [EMAIL_REDACTED] | linkedin.com/in/nadia-demo | github.com/nadia-demo

  Full address: [ADDRESS_REDACTED] | Date of birth: [DOB_REDACTED]


 Software engineering graduate focused on applied machine learning and LLM-powered products. Experienced in building
  retrieval workflows, Python APIs, and model evaluations for student and internship projects.

 Education Level

Institut Teknologi Nusantara - Bandung, Indonesia                            Aug 2021 - Jul 2025
 Bachelor of Computer Science, GPA 3.78/4.00

  • Relevant coursework: Machine Learning, Natural Language Processing, Database Systems.

  • Graduation project: Retrieval-based assistant for university policy FAQs.


 Organizational & Professional Experience

Awan Data Labs - Jakarta, Indonesia                                        Feb 2025 - Jul 2025
 AI Engineer Intern

  • Built a FastAPI service that retrieved relevant policy documents before generating answers using a hosted LLM.


## 2. Ekstraksi dan validasi bukti
### 2.1 Membentuk profil kandidat terstruktur
Mengirim `masked_cv` ke LLM melalui `extract_candidate`, lalu menampilkan profil Pydantic sebagai JSON. Struktur profil mencakup nama, pendidikan, skill, pengalaman, proyek, sertifikasi, dan `extraction_caveats`.

Perhatikan caveat saat membaca hasil: output terstruktur tetap perlu diperiksa terhadap CV sumber.

In [3]:
from mini_assignment_sekolahkarir.extractor import extract_candidate

candidate = extract_candidate(masked_cv)

print(candidate.model_dump_json(indent=2))

{
  "name": "NADIA PUTRI MAHARANI",
  "education": [
    {
      "institution": "Institut Teknologi Nusantara - Bandung, Indonesia",
      "degree": "Bachelor of Computer Science",
      "field_of_study": null,
      "graduation_year": 2025
    }
  ],
  "skills": [
    "Python",
    "FastAPI",
    "SQL",
    "scikit-learn",
    "RAG",
    "embeddings",
    "prompt engineering",
    "REST APIs",
    "Technical writing",
    "ownership",
    "collaboration",
    "analytical reasoning"
  ],
  "experience": [
    {
      "organization": "Awan Data Labs - Jakarta, Indonesia",
      "role": "AI Engineer Intern",
      "start_date": "Feb 2025",
      "end_date": "Jul 2025",
      "description": [
        "Built a FastAPI service that retrieved relevant policy documents before generating answers using a hosted LLM.",
        "Created a labeled evaluation set of 120 synthetic queries and tracked answer relevance and unsupported claims."
      ]
    },
    {
      "organization": "Campus AI Soci

### 2.2 Memeriksa dukungan pada teks sumber
`validate_evidence` memeriksa skill, nama institusi pendidikan, organisasi pengalaman, dan nama proyek. Pemeriksaan menggunakan pencarian substring setelah normalisasi kapitalisasi dan whitespace.

- ✅ Nilai ditemukan pada teks sumber yang diperiksa.
- ⚠️ Nilai belum didukung oleh pencarian tersebut dan perlu ditinjau.

Status ini tidak memverifikasi semua field, kebenaran faktual CV, atau kemampuan kandidat; variasi istilah juga dapat tidak terdeteksi.

In [4]:
from mini_assignment_sekolahkarir.validator import validate_evidence

evidence_checks = validate_evidence(candidate, masked_cv)

for check in evidence_checks:
    status = "✅" if check.supported else "⚠️"
    print(f"{status} {check.field}: {check.value}")

✅ skills: Python
✅ skills: FastAPI
✅ skills: SQL
✅ skills: scikit-learn
✅ skills: RAG
✅ skills: embeddings
✅ skills: prompt engineering
✅ skills: REST APIs
✅ skills: Technical writing
✅ skills: ownership
✅ skills: collaboration
✅ skills: analytical reasoning
✅ education.institution: Institut Teknologi Nusantara - Bandung, Indonesia
✅ experience.organization: Awan Data Labs - Jakarta, Indonesia
✅ experience.organization: Campus AI Society - Bandung, Indonesia
✅ projects.name: PolicyLens - RAG-Based Knowledge Assistant
✅ projects.name: Customer Ticket Triage


### 2.3 Menghitung evidence coverage
Menghitung proporsi pemeriksaan yang didukung: `supported / total`. Jika tidak ada pemeriksaan, nilai pada cell ini menjadi `0`.

Coverage menggambarkan dukungan tekstual pada field yang diperiksa, bukan akurasi seluruh ekstraksi atau nilai kecocokan kandidat.

In [5]:
total = len(evidence_checks)
supported = sum(check.supported for check in evidence_checks)

coverage = supported / total if total else 0

print(f"Evidence checks: {total}")
print(f"Supported: {supported}")
print(f"Coverage: {coverage:.1%}")

Evidence checks: 17
Supported: 17
Coverage: 100.0%


### 2.4 Percobaan penambahan skill
Membuat salinan mendalam profil dan menambahkan `Kubernetes`, lalu menampilkan hasil pemeriksaannya. Profil `candidate` asli tetap dipakai pada tahap berikutnya.

Percobaan ini membantu mengamati perilaku validator untuk skill tambahan. Cell hanya mencetak hasil; tidak ada assertion bahwa skill tersebut harus tidak didukung.

In [6]:
test_candidate = candidate.model_copy(deep=True)
test_candidate.skills.append("Kubernetes")

test_checks = validate_evidence(test_candidate, masked_cv)

for check in test_checks:
    if check.value == "Kubernetes":
        print(check)

EvidenceCheck(field='skills', value='Kubernetes', supported=False)


## 3. Ringkasan, pencocokan, dan rekomendasi
### 3.1 Membuat overview kandidat
Menghasilkan ringkasan melalui LLM berdasarkan profil dan evidence checks. Output dipisahkan menjadi ringkasan dan caveat agar informasi yang perlu dikonfirmasi mudah ditemukan.

Overview mendeskripsikan kandidat; pencocokan terhadap kebutuhan pekerjaan dilakukan pada tahap berikutnya.

In [7]:
from mini_assignment_sekolahkarir.overview import (
    generate_candidate_overview
)

overview = generate_candidate_overview(
    candidate,
    evidence_checks,
)

print("=== CANDIDATE OVERVIEW ===")
print(overview.summary)

print("\n=== CAVEATS ===")
for caveat in overview.caveats:
    print(f"- {caveat}")

=== CANDIDATE OVERVIEW ===
Nadia Putri Maharani is pursuing a Bachelor of Computer Science at Institut Teknologi Nusantara in Bandung, Indonesia, graduating in 2025. She possesses skills in Python, FastAPI, SQL, scikit-learn, RAG, embeddings, prompt engineering, REST APIs, technical writing, ownership, collaboration, and analytical reasoning. Nadia has interned as an AI Engineer at Awan Data Labs in Jakarta from February to July 2025, where she built a FastAPI service and created an evaluation set for tracking answer relevance. Additionally, she serves as a Machine Learning Coordinator at the Campus AI Society in Bandung, mentoring students in Python and machine learning. Her projects include 'PolicyLens - RAG-Based Knowledge Assistant', focusing on embedding search and citation-aware answer generation, and 'Customer Ticket Triage', where she trained a multi-class classifier and reported on its performance. Nadia holds an Introductory Machine Learning Certificate from a training provid

### 3.2 Mengimpor fungsi pencocokan
Cell berikut mengimpor `match_candidate`. Import ini juga muncul pada cell selanjutnya dan sengaja dipertahankan sesuai susunan kode asli.

In [8]:
from mini_assignment_sekolahkarir.matcher import match_candidate

### 3.3 Memuat kebutuhan pekerjaan dan menjalankan matching
Membaca `data/jobs/ai_engineer.json`, memvalidasinya sebagai `JobRequirements`, lalu membandingkannya dengan skill kandidat yang didukung evidence checks.

Konfigurasi saat dokumentasi dibuat: **Junior AI Engineer**, skill wajib Python, SQL, dan REST APIs; skill tambahan FastAPI, RAG, dan Docker; pengalaman minimum 0 tahun; `requirements_confirmed` bernilai `true`.

Pencocokan menggunakan kesamaan nama skill setelah normalisasi kapitalisasi dan whitespace. `unverified_*` berarti belum terverifikasi dari CV, bukan bukti kandidat tidak memiliki kemampuan tersebut.

In [9]:
import json
from pathlib import Path

from mini_assignment_sekolahkarir.schemas import JobRequirements
from mini_assignment_sekolahkarir.matcher import match_candidate

job_data = json.loads(
    Path("data/jobs/ai_engineer.json").read_text(encoding="utf-8")
)

job = JobRequirements.model_validate(job_data)

match_result = match_candidate(
    candidate=candidate,
    job=job,
    evidence_checks=evidence_checks,
)

print(match_result.model_dump_json(indent=2))

{
  "job_id": "JOB-001",
  "candidate_name": "NADIA PUTRI MAHARANI",
  "matched_required_skills": [
    "Python",
    "SQL",
    "REST APIs"
  ],
  "unverified_required_skills": [],
  "matched_preferred_skills": [
    "FastAPI",
    "RAG"
  ],
  "unverified_preferred_skills": [
    "Docker"
  ],
  "experience_requirement": 0,
  "experience_status": "not_required",
  "caveats": [
    "Some preferred skills could not be verified from the candidate's CV."
  ]
}


### 3.4 Menghasilkan rekomendasi untuk peninjau
Menggabungkan overview dan hasil matching untuk menghasilkan penjelasan, strengths, caveat, dan tindak lanjut melalui LLM.

`review_recommended` dan `manual_review_required` merupakan status peninjauan. Implementasi mewajibkan human review, mempertahankan caveat, dan menetapkan `manual_review_required` ketika skill wajib belum terverifikasi atau durasi pengalaman masih perlu ditinjau.

In [10]:
from mini_assignment_sekolahkarir.recommendation import (
    generate_recommendation
)

recommendation = generate_recommendation(
    overview=overview,
    match_result=match_result,
)

print(recommendation.model_dump_json(indent=2))

{
  "status": "review_recommended",
  "reasoning": "Nadia possesses several required and preferred skills that align with the job requirements. She has documented experience with Python, SQL, and REST APIs, which are essential for the position. Additionally, her skills in FastAPI and RAG make her a strong match for the preferred qualifications. As there are some unverified preferred skills (specifically Docker), further verification could be beneficial, but she meets the core requirements for the role.",
  "strengths": [
    "Proficient in Python, as indicated by her coursework and internship experience.",
    "Experienced with SQL, showcasing the ability to manage databases and data queries effectively.",
    "Skilled in REST APIs development through her internship where she built FastAPI services.",
    "Hands-on experience with FastAPI, a preferred skill for the job, through her internship project.",
    "Knowledge of RAG technology, demonstrated in her project 'PolicyLens - RAG-Bas

### 3.5 Memeriksa aturan rekomendasi
Assertion memeriksa status yang diperbolehkan, kewajiban human review, reasoning yang tidak kosong, penerusan caveat dari matching, serta status manual review ketika skill wajib belum terverifikasi.

Pesan sukses muncul jika semua assertion pada cell ini lolos; cakupannya terbatas pada aturan tersebut.

In [11]:
assert recommendation.status in {
    "review_recommended",
    "manual_review_required",
}

assert recommendation.requires_human_review is True

assert len(recommendation.reasoning.strip()) > 0

for caveat in match_result.caveats:
    assert caveat in recommendation.caveats

if match_result.unverified_required_skills:
    assert recommendation.status == "manual_review_required"

print("✅ Recommendation safety checks passed!")

✅ Recommendation safety checks passed!


## 4. Evaluasi pada lima CV
### 4.1 Menjalankan pipeline secara batch
Mengambil seluruh PDF di `data/cv/` dan mengharuskan jumlahnya tepat lima. Setiap CV melalui parsing, masking, ekstraksi, validasi bukti, overview, matching terhadap `job` yang sama, dan rekomendasi.

Hasil disimpan dalam `evaluation_results`. Error per CV dicatat agar pemrosesan CV lain dapat berlanjut. Status `success` menunjukkan pipeline selesai, bukan bahwa seluruh hasilnya benar.

In [13]:
from pathlib import Path

from mini_assignment_sekolahkarir.parser import extract_text_from_pdf
from mini_assignment_sekolahkarir.privacy import mask_personal_information
from mini_assignment_sekolahkarir.extractor import extract_candidate
from mini_assignment_sekolahkarir.validator import validate_evidence
from mini_assignment_sekolahkarir.overview import generate_candidate_overview
from mini_assignment_sekolahkarir.matcher import match_candidate
from mini_assignment_sekolahkarir.recommendation import generate_recommendation

cv_files = sorted(Path("data/cv").glob("*.pdf"))

assert len(cv_files) == 5, (
    f"Expected 5 CVs, found {len(cv_files)}"
)

evaluation_results = {}

for pdf_path in cv_files:
    print(f"Processing: {pdf_path.name}")

    try:
        # 1. Parse PDF
        raw_text = extract_text_from_pdf(pdf_path)

        # 2. Privacy masking
        masked_text = mask_personal_information(raw_text)

        # 3. LLM extraction
        profile = extract_candidate(masked_text)

        # 4. Evidence validation
        checks = validate_evidence(profile, masked_text)

        # 5. Candidate overview
        candidate_overview = generate_candidate_overview(
            profile, checks
        )

        # 6. Job matching
        match = match_candidate(profile, job, checks)

        # 7. Recommendation
        recommendation_result = generate_recommendation(
            candidate_overview, match
        )

        evaluation_results[pdf_path.name] = {
            "success": True,
            "profile": profile,
            "evidence_checks": checks,
            "overview": candidate_overview,
            "match": match,
            "recommendation": recommendation_result,
        }

        print("  ✅ Success")

    except Exception as error:
        evaluation_results[pdf_path.name] = {
            "success": False,
            "error": str(error),
        }
        print(f"  ❌ {type(error).__name__}: {error}")

Processing: CV_01_Nadia_Putri_AI_Engineer.pdf
  ✅ Success
Processing: CV_02_Rizky_Pratama_Frontend.pdf
  ✅ Success
Processing: CV_03_Siti_Rahma_Data_Analyst.pdf
  ✅ Success
Processing: CV_04_Bima_Santoso_Backend.pdf
  ✅ Success
Processing: CV_05_Alya_Fauziah_Ambiguous.pdf
  ✅ Success


### 4.2 Merangkum hasil batch
Tabel `df_results` menampilkan:

| Kolom | Makna |
| --- | --- |
| `cv` | Nama file yang diproses |
| `pipeline_success` | Apakah semua tahap pipeline selesai untuk CV tersebut |
| `evidence_coverage` | Proporsi evidence checks yang didukung, atau nilai kosong jika gagal/tidak ada pemeriksaan |
| `human_review` | Nilai kewajiban peninjauan dari rekomendasi |

Gunakan tabel ini untuk melihat hasil operasional dan dukungan tekstual; tabel ini tidak memberi peringkat kandidat.

In [15]:
import pandas as pd

rows = []

for filename, result in evaluation_results.items():
    if not result["success"]:
        rows.append({
            "cv": filename,
            "pipeline_success": False,
            "evidence_coverage": None,
            "human_review": None,
        })
        continue

    checks = result["evidence_checks"]

    supported = sum(check.supported for check in checks)
    coverage = supported / len(checks) if checks else None

    rows.append({
        "cv": filename,
        "pipeline_success": True,
        "evidence_coverage": coverage,
        "human_review": (
            result["recommendation"].requires_human_review
        ),
    })

df_results = pd.DataFrame(rows)

display(df_results)

,cv,pipeline_success,evidence_coverage,human_review
0,CV_01_Nadia_Putri_AI_Engineer.pdf,True,1.0,True
1,CV_02_Rizky_Pratama_Frontend.pdf,True,1.0,True
2,CV_03_Siti_Rahma_Data_Analyst.pdf,True,1.0,True
3,CV_04_Bima_Santoso_Backend.pdf,True,1.0,True
4,CV_05_Alya_Fauziah_Ambiguous.pdf,True,1.0,True


### 4.3 Mengukur ekstraksi skill untuk CV Nadia
Membandingkan skill hasil ekstraksi batch CV Nadia dengan delapan skill referensi yang ditulis langsung di cell.

- **Precision:** proporsi skill prediksi yang sesuai referensi.
- **Recall:** proporsi skill referensi yang ditemukan dalam prediksi.
- **F1:** rata-rata harmonik precision dan recall.
- **Unexpected / missed:** skill prediksi di luar referensi / skill referensi yang tidak ditemukan.

Perhitungan berbasis himpunan setelah normalisasi kapitalisasi dan whitespace. Metrik ini hanya mencakup skill satu CV dengan daftar referensi tersebut; skill di luar referensi belum tentu salah. Cell bergantung pada keberhasilan hasil batch Nadia.

In [17]:
from mini_assignment_sekolahkarir.evaluation import (
    calculate_extraction_metrics
)

nadia_file = "CV_01_Nadia_Putri_AI_Engineer.pdf"

expected_skills_nadia = [
    "Python",
    "FastAPI",
    "SQL",
    "scikit-learn",
    "RAG",
    "embeddings",
    "prompt engineering",
    "REST APIs",
]

predicted_skills_nadia = (
    evaluation_results[nadia_file]["profile"].skills
)

metrics = calculate_extraction_metrics(
    predicted=predicted_skills_nadia,
    expected=expected_skills_nadia,
)

print(metrics)

{'precision': 0.667, 'recall': 1.0, 'f1': 0.8, 'true_positives': 8, 'false_positives': 4, 'false_negatives': 0, 'unexpected': ['analytical reasoning', 'collaboration', 'ownership', 'technical writing'], 'missed': []}


## 5. Pemeriksaan privasi dan aturan tambahan
### 5.1 Memeriksa contoh privacy masking
Menjalankan empat contoh sintetis: email, telepon dengan spasi, alamat lengkap, dan NIK. Sebuah contoh dinyatakan lolos jika placeholder yang diharapkan muncul dalam output.

Pass rate berlaku untuk empat contoh ini saja. Pemeriksaan tidak mencakup seluruh variasi data pribadi, dan belum memeriksa secara eksplisit bahwa seluruh nilai sumber sudah hilang.

In [18]:
from mini_assignment_sekolahkarir.privacy import (
    mask_personal_information
)

privacy_test_cases = [
    {
        "name": "Email",
        "input": "Email: test@example.com",
        "expected": "[EMAIL_REDACTED]",
    },
    {
        "name": "Phone with spaces",
        "input": "Phone: +62 800 0000 1001",
        "expected": "[PHONE_REDACTED]",
    },
    {
        "name": "Full address",
        "input": (
            "Full address: Jl. Anggrek Fiktif No. 18, "
            "RT 003/RW 008, Depok, 16412"
        ),
        "expected": "[ADDRESS_REDACTED]",
    },
    {
        "name": "NIK",
        "input": "NIK: 3271010101010001",
        "expected": "[NIK_REDACTED]",
    },
]

privacy_results = []

for case in privacy_test_cases:
    output = mask_personal_information(case["input"])
    passed = case["expected"] in output

    privacy_results.append({
        "test": case["name"],
        "passed": passed,
    })

    print(
        f"{'✅' if passed else '❌'} "
        f"{case['name']}: {output}"
    )

passed_count = sum(r["passed"] for r in privacy_results)

print(
    f"\nPrivacy masking pass rate: "
    f"{passed_count}/{len(privacy_results)}"
)

✅ Email: Email: [EMAIL_REDACTED]
✅ Phone with spaces: Phone: [PHONE_REDACTED]
✅ Full address: Full address: [ADDRESS_REDACTED]
✅ NIK: NIK: [NIK_REDACTED]

Privacy masking pass rate: 4/4


### 5.2 Memeriksa aturan rekomendasi pada hasil batch
Untuk setiap CV yang pipeline-nya berhasil, tabel berikut memeriksa human review, penerusan caveat matching, dan status manual review jika skill wajib belum terverifikasi.

CV yang gagal dilewati pada tabel ini; lihat kembali `df_results` untuk mengetahui kegagalan pipeline. Cell menampilkan boolean pemeriksaan tanpa assertion tambahan.

In [19]:
safety_results = []

for filename, result in evaluation_results.items():
    if not result["success"]:
        continue

    recommendation = result["recommendation"]
    match = result["match"]

    human_review_ok = (
        recommendation.requires_human_review is True
    )

    caveats_ok = all(
        caveat in recommendation.caveats
        for caveat in match.caveats
    )

    status_ok = (
        not match.unverified_required_skills
        or recommendation.status == "manual_review_required"
    )

    safety_results.append({
        "cv": filename,
        "human_review": human_review_ok,
        "caveats_preserved": caveats_ok,
        "status_rule": status_ok,
    })

display(pd.DataFrame(safety_results))

,cv,human_review,caveats_preserved,status_rule
0,CV_01_Nadia_Putri_AI_Engineer.pdf,True,True,True
1,CV_02_Rizky_Pratama_Frontend.pdf,True,True,True
2,CV_03_Siti_Rahma_Data_Analyst.pdf,True,True,True
3,CV_04_Bima_Santoso_Backend.pdf,True,True,True
4,CV_05_Alya_Fauziah_Ambiguous.pdf,True,True,True


### 5.3 Menolak kebutuhan pekerjaan yang belum dikonfirmasi
Membuat salinan `job` dengan `requirements_confirmed=False`, lalu mencoba matching. `ValueError` yang ditangkap menunjukkan aturan konfirmasi dijalankan; jika matching tetap berlanjut, cell memunculkan `AssertionError`.

Objek `job` asli tetap digunakan seperti sebelumnya.

In [20]:
unconfirmed_job = job.model_copy(
    update={"requirements_confirmed": False}
)

try:
    match_candidate(
        candidate,
        unconfirmed_job,
        evidence_checks,
    )
except ValueError as error:
    print("✅ Unconfirmed job correctly blocked:", error)
else:
    raise AssertionError(
        "Unconfirmed job should not proceed to matching."
    )

✅ Unconfirmed job correctly blocked: Job requirements must be confirmed by HR.


### 5.4 Cell kosong dari notebook asli
Cell code kosong berikut dipertahankan agar jumlah dan urutan seluruh cell code tetap sama dengan `demo.ipynb`.

## Catatan interpretasi

Notebook ini memperlihatkan alur dan pemeriksaan demo pada data fiktif. Baca hasil ekstraksi bersama evidence checks dan caveat; validasi substring serta metrik satu CV memiliki cakupan terbatas. Rekomendasi tetap memerlukan peninjauan manusia.

Untuk memahami implementasi lebih lanjut, lihat modul `parser.py`, `privacy.py`, `extractor.py`, `validator.py`, `overview.py`, `matcher.py`, `recommendation.py`, `evaluation.py`, dan definisi struktur data di `schemas.py`, semuanya di `src/mini_assignment_sekolahkarir/`.